# Prétraitement — toutes les visites (V0, V1, V3, V5)

Exécutez ce notebook de haut en bas.  
Chaque section `### Vx` traite le fichier `Output/Vx.xlsx` et écrit le résultat dans `Output2/Vx.xlsx`.

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

---
## Fonction générique de traitement d'une visite

Cette fonction reçoit le **numéro de visite** (entier : 0, 1, 3 ou 5)  
et retourne un dictionnaire `{nom_feuille: DataFrame}` prêt à être écrit dans Excel.

In [38]:
def traiter_visite(v, vc=False):
    """
    Lit Output/V{v}.xlsx et reproduit exactement le même traitement
    que celui développé pour V0.

    Paramètre
    ---------
    v : int ou str
        Numéro de visite (0, 1, 3 ou 5) ou "c" pour VC.
    vc : bool
        Si True, applique uniquement le traitement VC.

    Retourne
    --------
    dict {str: pd.DataFrame}
        Clés = noms de feuilles, valeurs = DataFrames traités.
    """
    chemin = f"Output/v{v}.xlsx"

    # ------------------------------------------------------------------
    # Feuille 1  →  DATES_VISITE  (commun à tous)
    # ------------------------------------------------------------------
    df_dates = pd.read_excel(chemin, sheet_name="DATES_VISITE")
    print(f"\nFeuille DATES_VISITE : {df_dates.shape}")

    # ------------------------------------------------------------------
    # Feuille 3  →  df_LEED  (commun à tous)
    # ------------------------------------------------------------------
    df_feuil1 = pd.read_excel(chemin, sheet_name="Feuil1")
    print(f"\nFeuille Feuil1 (LEED) : {df_feuil1.shape}")

    df_feuil1.rename(
        columns={"Subject Identifier for the Study": "SUBJID"}, inplace=True
    )
    df_LEED = df_feuil1.copy()
    df_LEED.drop("Num", axis=1, inplace=True)
    df_LEED.insert(0, "VISITE", v)
    print(f"df_LEED              : {df_LEED.shape}")

    # ------------------------------------------------------------------
    # Feuille 4  →  df_LEED_info  (commun à tous)
    # ------------------------------------------------------------------
    df_ledd = pd.read_excel(chemin, sheet_name="LEDD")
    print(f"\nFeuille LEDD         : {df_ledd.shape}")

    df_LEED_info = df_ledd.iloc[:, :-7].copy()
    df_LEED_info.drop("v", axis=1, inplace=True)
    df_LEED_info.rename(
        columns={"Subject Identifier for the Study": "SUBJID"}, inplace=True
    )
    df_LEED_info.insert(0, "VISITE", v)
    df_LEED_info.drop("Num", axis=1, inplace=True)
    df_LEED_info.drop("Visit", axis=1, inplace=True)
    print(f"df_LEED_info         : {df_LEED_info.shape}")

    # ------------------------------------------------------------------
    # Feuille 5  →  df_PSYCHOTROPES  (commun à tous)
    # ------------------------------------------------------------------
    df_PSYCHOTROPES = pd.read_excel(chemin, sheet_name="PSYCHOTROPES")
    print(f"\nFeuille PSYCHOTROPES : {df_PSYCHOTROPES.shape}")

    df_PSYCHOTROPES.insert(0, "VISITE", v)
    df_PSYCHOTROPES.drop(columns=["NUM", "Unnamed: 3"], inplace=True)
    df_PSYCHOTROPES.drop(columns=["VISIT", "INIT_PAT"], inplace=True)
    print(f"df_PSYCHOTROPES      : {df_PSYCHOTROPES.shape}")

    # ------------------------------------------------------------------
    # Feuille 6  →  df_AUTRE_PARKINSON  (commun à tous)
    # ------------------------------------------------------------------
    df_AUTRE_PARKINSON = pd.read_excel(chemin, sheet_name="AUTRE_PARKINSON")
    print(f"\nFeuille AUTRE_PARKINSON : {df_AUTRE_PARKINSON.shape}")

    df_AUTRE_PARKINSON.insert(0, "VISITE", v)
    df_AUTRE_PARKINSON.drop(columns=["NUM", "Unnamed: 3"], inplace=True)
    df_AUTRE_PARKINSON.drop(columns=["VISIT", "INIT_PAT"], inplace=True)
    print(f"df_AUTRE_PARKINSON   : {df_AUTRE_PARKINSON.shape}")

    # ------------------------------------------------------------------
    # Feuille 7  →  df_CONSO_SPECIFIQUE  (commun à tous)
    # ------------------------------------------------------------------
    df_CONSO_SPECIFIQUE = pd.read_excel(chemin, sheet_name="CONSO_SPECIFIQUE")
    print(f"\nFeuille CONSO_SPECIFIQUE : {df_CONSO_SPECIFIQUE.shape}")

    df_CONSO_SPECIFIQUE.insert(0, "VISITE", v)
    df_CONSO_SPECIFIQUE.drop(
        columns=["NUM", "VISIT_NOM", "NUM_CENTRE", "NUM_PAT", "INIT_PAT"],
        inplace=True,
    )
    df_CONSO_SPECIFIQUE.drop("VISIT", axis=1, inplace=True)
    print(f"df_CONSO_SPECIFIQUE  : {df_CONSO_SPECIFIQUE.shape}")

    # ------------------------------------------------------------------
    # Retour VC  →  seulement les feuilles communes
    # ------------------------------------------------------------------
    if vc:
        return {
            "Vc"     : df_dates,
            "LEED_info"        : df_LEED_info,
            "LEED"             : df_LEED,
            "CONSO_SPECIFIQUE" : df_CONSO_SPECIFIQUE,
            "PSYCHOTROPES"     : df_PSYCHOTROPES,
            "AUTRE_PARKINSON"  : df_AUTRE_PARKINSON,
        }

    # ------------------------------------------------------------------
    # Suite  →  uniquement pour V0 à V5
    # ------------------------------------------------------------------
    df_base = pd.read_excel(chemin, sheet_name=f"df_v{v}")
    df_base.drop(columns=["POIDS_NR", "TAILLE_NR", "TITRE"], inplace=True)
    df_V = pd.merge(df_dates, df_base, on="SUBJID", how="left")
    df_V.insert(0, "VISITE", v)
    df_V.drop("INIT_PAT", axis=1, inplace=True)
    print(f"df_V (après fusion)  : {df_V.shape}")

    df_DIGITSMT = pd.read_excel(chemin, sheet_name="DIGITSMT_TRAILMT_DKEFS")
    print(f"\nFeuille DIGITSMT_TRAILMT_DKEFS : {df_DIGITSMT.shape}")
    df_DIGITSMT.insert(0, "VISITE", v)
    df_DIGITSMT.drop(columns=["TITRE", "INIT_PAT"], inplace=True)

    df_PDQ39 = pd.read_excel(chemin, sheet_name="PDQ39")
    df_PDQ39.insert(0, "VISITE", v)
    df_PDQ39.drop(columns=["VISIT", "INIT_PAT"], inplace=True)

    df_LARS = pd.read_excel(chemin, sheet_name="LARS")
    df_LARS.insert(0, "VISITE", v)
    df_LARS.drop(columns=["TITRE", "INIT_PAT"], inplace=True)

    df_HAMA = pd.read_excel(chemin, sheet_name="HAMA")
    df_HAMA.insert(0, "VISITE", v)
    df_HAMA.drop(columns=["TITRE", "INIT_PAT"], inplace=True)

    df_HAMD = pd.read_excel(chemin, sheet_name="HAMD")
    df_HAMD.insert(0, "VISITE", v)
    df_HAMD.drop(columns=["TITRE", "INIT_PAT"], inplace=True)

    df_MOCA = pd.read_excel(chemin, sheet_name="MOCA")
    df_MOCA.insert(0, "VISITE", v)
    df_MOCA.drop(columns=["TITRE", "INIT_PAT"], inplace=True)

    df_QUIP = pd.read_excel(chemin, sheet_name="QUIP")
    df_QUIP.insert(0, "VISITE", v)
    df_QUIP.drop(columns=["TITRE", "INIT_PAT"], inplace=True)

    df_ECMP = pd.read_excel(chemin, sheet_name="ECMP")
    df_ECMP.insert(0, "VISITE", v)
    df_ECMP.drop(columns=["TITRE", "INIT_PAT"], inplace=True)

    df_UPDRS = pd.read_excel(chemin, sheet_name="UPDRS")
    df_UPDRS.insert(0, "VISITE", v)
    df_UPDRS.drop(columns=["TITRE", "INIT_PAT"], inplace=True)

    if v in (3, 5):
        df_UPDRSIII_COMPLET = pd.read_excel(chemin, sheet_name="UPDRSIII_COMPLET")
        df_UPDRSIII_COMPLET.insert(0, "VISITE", v)
        df_UPDRSIII_COMPLET.drop(columns=["TITRE", "INIT_PAT"], inplace=True)

    # ------------------------------------------------------------------
    # Retour V0 à V5
    # ------------------------------------------------------------------
    result = {
        f"V{v}"                  : df_V,
        "LEED_info"              : df_LEED_info,
        "LEED"                   : df_LEED,
        "CONSO_SPECIFIQUE"       : df_CONSO_SPECIFIQUE,
        "PSYCHOTROPES"           : df_PSYCHOTROPES,
        "AUTRE_PARKINSON"        : df_AUTRE_PARKINSON,
        "UPDRSIV"                : df_UPDRS,
        "PDQ39"                  : df_PDQ39,
        "QUIP"                   : df_QUIP,
        "MOCA"                   : df_MOCA,
        "HAMA"                   : df_HAMA,
        "HAMD"                   : df_HAMD,
        "LARS"                   : df_LARS,
        "ECMP"                   : df_ECMP,
        "DIGITSMT_TRAILMT_DKEFS" : df_DIGITSMT,
    }

    if v in (3, 5):
        result["UPDRSIII"] = df_UPDRSIII_COMPLET

    return result

In [34]:
ORDRE_FEUILLES = [
    "LEED_info",
    "LEED",
    "CONSO_SPECIFIQUE",
    "PSYCHOTROPES",
    "AUTRE_PARKINSON",
    "UPDRSIII",          
    "UPDRSIII_TOTAUX",   
    "UPDRSIV",
    "PDQ39",
    "QUIP",
    "MOCA",
    "HAMA",
    "HAMD",
    "LARS",
    "ECMP",
    "DIGITSMT_TRAILMT_DKEFS",
]

def reordonner_feuilles(sheets, v):
    cle_visite    = f"V{v}"
    ordre_complet = [cle_visite] + ORDRE_FEUILLES
    return {k: sheets[k] for k in ordre_complet if k in sheets}

---
## UPDRS III — données partagées entre V0 et V1

Le fichier `Data/Matthieu_Soumaya_Dec2025.xlsx` contient deux onglets communs :
- `UPDRSIII_COMPLET_V0_V1`  
- `UPDRSIII_TOTAUX`

In [8]:
def charger_updrs_iii():
    """
    Lit le fichier UPDRS III et sépare les colonnes V0 / V1.
    Retourne (df_v0_UPDRS_III, df_v1_UPDRS_III,
              df_v0_UPDRSIII_TOTAUX, df_v1_UPDRSIII_TOTAUX,
              df_v3_UPDRSIII_TOTAUX, df_v5_UPDRSIII_TOTAUX)
    """
    chemin = "Data/Matthieu_Soumaya_Dec2025.xlsx"

    # ---- UPDRSIII complet ----
    df_III = pd.read_excel(chemin, sheet_name="UPDRSIII_COMPLET_V0_V1")
    # print(f"UPDRSIII_COMPLET_V0_V1 : {df_III.shape}")

    # Colonnes 1-263 → V0 ;  colonnes 264+ → V1
    df_v0_III = pd.concat([df_III.iloc[:, [0]], df_III.iloc[:, 1:264]], axis=1)
    df_v1_III = pd.concat([df_III.iloc[:, [0]], df_III.iloc[:, 264:]], axis=1)
    df_v0_III.insert(0, "VISITE", 0)
    df_v1_III.insert(0, "VISITE", 1)

    # ---- UPDRSIII totaux ----
    df_tot = pd.read_excel(chemin, sheet_name="UPDRSIII_TOTAUX ")
    print(f"UPDRSIII_TOTAUX        : {df_tot.shape}")

    # Repérage dynamique de la colonne séparatrice "Unnamed: 8"
    sep_col = df_tot.columns.get_loc("Unnamed: 8")  # = 8

    df_v0_tot  = pd.concat([df_tot.iloc[:, [0]], df_tot.iloc[:, 1:sep_col]], axis=1)
    df_v1_tot  = pd.concat([df_tot.iloc[:, [0]], df_tot.iloc[:, sep_col+1:sep_col+5]], axis=1)
    df_v3_tot  = df_tot[["SUBJID", "ON_TOTAL_V3"]].copy()
    df_v5_tot  = df_tot[["SUBJID", "ON_TOTAL_V5"]].copy()


    df_v0_tot.insert(0, "VISITE", 0) 
    df_v1_tot.insert(0, "VISITE", 1)  
    df_v3_tot.insert(0, "VISITE", 3)  
    df_v5_tot.insert(0, "VISITE", 5)  

    return df_v0_III, df_v1_III, df_v0_tot, df_v1_tot, df_v3_tot, df_v5_tot


df_v0_UPDRS_III, df_v1_UPDRS_III, \
df_v0_UPDRSIII_TOTAUX, df_v1_UPDRSIII_TOTAUX, \
df_v3_UPDRSIII_TOTAUX, df_v5_UPDRSIII_TOTAUX = charger_updrs_iii()

UPDRSIII_TOTAUX        : (836, 17)


---
## Traitement + écriture de chaque visite

### V0

In [36]:
sheets_V0 = traiter_visite(0)

# Ajout des feuilles UPDRS III propres à V0
sheets_V0["UPDRSIII"]        = df_v0_UPDRS_III
sheets_V0["UPDRSIII_TOTAUX"] = df_v0_UPDRSIII_TOTAUX
sheets_V0 = reordonner_feuilles(sheets_V0, 0)
# Écriture
OUTPUT_DIR = "Output2"
with pd.ExcelWriter(f"{OUTPUT_DIR}/V0.xlsx", engine="openpyxl") as writer:
    for sheet_name, df in sheets_V0.items():
        df.to_excel(writer, sheet_name=sheet_name[:31], index=False)
print(f"[OK] Output2/V0.xlsx  →  {len(sheets_V0)} feuilles")


Feuille DATES_VISITE : (836, 2)

Feuille Feuil1 (LEED) : (794, 7)
df_LEED              : (794, 7)

Feuille LEDD         : (794, 37)
df_LEED_info         : (794, 28)

Feuille PSYCHOTROPES : (835, 83)
df_PSYCHOTROPES      : (835, 80)

Feuille AUTRE_PARKINSON : (835, 83)
df_AUTRE_PARKINSON   : (835, 80)

Feuille CONSO_SPECIFIQUE : (835, 113)
df_CONSO_SPECIFIQUE  : (835, 108)
df_V (après fusion)  : (836, 5)

Feuille DIGITSMT_TRAILMT_DKEFS : (835, 22)
[OK] Output2/V0.xlsx  →  17 feuilles


### V1

In [39]:
sheets_V1 = traiter_visite(1)

# Ajout des feuilles UPDRS III propres à V1
sheets_V1["UPDRSIII"]        = df_v1_UPDRS_III
sheets_V1["UPDRSIII_TOTAUX"] = df_v1_UPDRSIII_TOTAUX
sheets_V1 = reordonner_feuilles(sheets_V1, 1)
OUTPUT_DIR = "Output2"
with pd.ExcelWriter(f"{OUTPUT_DIR}/V1.xlsx", engine="openpyxl") as writer:
    for sheet_name, df in sheets_V1.items():
        df.to_excel(writer, sheet_name=sheet_name[:31], index=False)
print(f"[OK] Output2/V1.xlsx  →  {len(sheets_V1)} feuilles")


Feuille DATES_VISITE : (836, 2)

Feuille Feuil1 (LEED) : (525, 7)
df_LEED              : (525, 7)

Feuille LEDD         : (525, 37)
df_LEED_info         : (525, 28)

Feuille PSYCHOTROPES : (835, 83)
df_PSYCHOTROPES      : (835, 80)

Feuille AUTRE_PARKINSON : (835, 83)
df_AUTRE_PARKINSON   : (835, 80)

Feuille CONSO_SPECIFIQUE : (835, 113)
df_CONSO_SPECIFIQUE  : (835, 108)
df_V (après fusion)  : (836, 6)

Feuille DIGITSMT_TRAILMT_DKEFS : (835, 22)
[OK] Output2/V1.xlsx  →  17 feuilles


### V3

In [19]:
sheets_V3 = traiter_visite(3)

# UPDRS III totaux pour V3 (colonne ON_TOTAL_V3 uniquement)
sheets_V3["UPDRSIII_TOTAUX"] = df_v3_UPDRSIII_TOTAUX
sheets_V3 = reordonner_feuilles(sheets_V3, 3)
OUTPUT_DIR = "Output2"
with pd.ExcelWriter(f"{OUTPUT_DIR}/V3.xlsx", engine="openpyxl") as writer:
    for sheet_name, df in sheets_V3.items():
        df.to_excel(writer, sheet_name=sheet_name[:31], index=False)
print(f"[OK] Output2/V3.xlsx  →  {len(sheets_V3)} feuilles")


Feuille DATES_VISITE : (836, 2)
df_V (après fusion)  : (836, 6)

Feuille Feuil1 (LEED) : (272, 7)
df_LEED              : (272, 7)

Feuille LEDD         : (272, 37)
df_LEED_info         : (272, 28)

Feuille PSYCHOTROPES : (835, 83)
df_PSYCHOTROPES      : (835, 80)

Feuille AUTRE_PARKINSON : (835, 83)
df_AUTRE_PARKINSON   : (835, 80)

Feuille CONSO_SPECIFIQUE : (835, 113)
df_CONSO_SPECIFIQUE  : (835, 108)

Feuille DIGITSMT_TRAILMT_DKEFS : (835, 22)
df_DIGITSMT          : (835, 21)

Feuille PDQ39        : (835, 45)
df_PDQ39             : (835, 44)

Feuille LARS         : (835, 14)
df_LARS              : (835, 13)

Feuille HAMA         : (835, 19)
df_HAMA              : (835, 18)

Feuille HAMD         : (835, 23)
df_HAMD              : (835, 22)

Feuille MOCA         : (835, 15)
df_MOCA              : (835, 14)

Feuille QUIP         : (835, 34)
df_QUIP              : (835, 33)

Feuille ECMP         : (835, 45)
df_ECMP              : (835, 44)

Feuille UPDRS        : (835, 10)
df_UPDRS_IV 

### V5

In [40]:
sheets_V5 = traiter_visite(5)

# UPDRS III totaux pour V5 (colonne ON_TOTAL_V5 uniquement)
sheets_V5["UPDRSIII_TOTAUX"] = df_v5_UPDRSIII_TOTAUX
sheets_V5 = reordonner_feuilles(sheets_V5, 5)
OUTPUT_DIR = "Output2"
with pd.ExcelWriter(f"{OUTPUT_DIR}/V5.xlsx", engine="openpyxl") as writer:
    for sheet_name, df in sheets_V5.items():
        df.to_excel(writer, sheet_name=sheet_name[:31], index=False)
print(f"[OK] Output2/V5.xlsx  →  {len(sheets_V5)} feuilles")


Feuille DATES_VISITE : (836, 2)

Feuille Feuil1 (LEED) : (313, 7)
df_LEED              : (313, 7)

Feuille LEDD         : (313, 37)
df_LEED_info         : (313, 28)

Feuille PSYCHOTROPES : (835, 83)
df_PSYCHOTROPES      : (835, 80)

Feuille AUTRE_PARKINSON : (835, 83)
df_AUTRE_PARKINSON   : (835, 80)

Feuille CONSO_SPECIFIQUE : (835, 113)
df_CONSO_SPECIFIQUE  : (835, 108)
df_V (après fusion)  : (836, 6)

Feuille DIGITSMT_TRAILMT_DKEFS : (835, 22)
[OK] Output2/V5.xlsx  →  17 feuilles


### Vc 

In [35]:
sheets_Vc = traiter_visite("c", vc=True)
sheets_Vc = reordonner_feuilles(sheets_Vc, "c")

OUTPUT_DIR = "Output2"
with pd.ExcelWriter(f"{OUTPUT_DIR}/Vc.xlsx", engine="openpyxl") as writer:
    for sheet_name, df in sheets_Vc.items():
        df.to_excel(writer, sheet_name=sheet_name[:31], index=False)
print(f"[OK] Output2/Vc.xlsx  →  {len(sheets_Vc)} feuilles")


Feuille DATES_VISITE : (836, 2)

Feuille Feuil1 (LEED) : (491, 7)
df_LEED              : (491, 7)

Feuille LEDD         : (491, 37)
df_LEED_info         : (491, 28)

Feuille PSYCHOTROPES : (835, 83)
df_PSYCHOTROPES      : (835, 80)

Feuille AUTRE_PARKINSON : (835, 83)
df_AUTRE_PARKINSON   : (835, 80)

Feuille CONSO_SPECIFIQUE : (835, 113)
df_CONSO_SPECIFIQUE  : (835, 108)
[OK] Output2/Vc.xlsx  →  6 feuilles


# Prétraitement info statiques 

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
xl = pd.ExcelFile("Output/version_1/info_statiques.xlsx")

print(f"Nombre de feuille : ",len(xl.sheet_names))
# une par ligne
for nom in xl.sheet_names:
    print(nom)

Nombre de feuille :  2
Sheet1
Sheet2


In [6]:
df_feuille1 = pd.read_excel("Output/version_1/info_statiques.xlsx", sheet_name="Sheet1")
print(df_feuille1.shape)
df_feuille1.head()

(836, 15)


,SUBJID,INIT_PAT,D_SCREEN,D_1ER_SYMPT,D_DIAG,D_LDOPA,D_TTT_DOPAM,D_FLUCTU_MOTR,D_FLUCTU_NONMOTR,D_DYSKINESIE,Mut_GBA,alias_GBA,DDN,AGE,SEXE
0,Subject Identifier for the Study,initiales patient,date de la visite de screening,année premiers symptomes,année du diagnostic de la maladie,année d'introduction de la L-DOPA,année d'introduction du traitement dopaminergique,année d'apparition des fluctuations motrices,année d'apparition des fluctuations non motrices,année d'apparition des dyskinésies,NaN,NaN,date de naissance,âge,sexe
1,01-001,SR,18/11/2013,1998,1999,2000,2000,2002,2000,.D,NaN,NaN,02/1946,67,1
2,01-002,TM,13/01/2014,2006,2006,2006,2006,2011,.K,.K,NaN,NaN,01/1954,59,1
3,01-003,SJ,04/03/2014,2001,2001,2003,2001,2004,.K,2004,NaN,NaN,02/1953,61,1
4,01-004,DJ,12/05/2014,1998,2000,2003,2000,2003,2003,2003,NaN,NaN,08/1948,65,2


In [11]:
df_feuille2 = pd.read_excel("Output/version_1/info_statiques.xlsx", sheet_name="Sheet2")
print(df_feuille2.shape)
df_feuille2.head()

(836, 9)


,SUBJID,SUIVI_ETUDE,D_FIN_ETUDE,D_SORTIE_PREMA,MOTIF_SORTIE_PREMA,AUTRE_PRECIS,D_INVESTIGATEUR,NOM_INVESTIGATEUR,MOTIF_SORTIE_PREMA1
0,Subject Identifier for the Study,suivi étude,date de fin de l'étude,date de sortie prématurée,motif de sortie prématurée,autre précision,date de signature de l'investigateur,nom de l'investigateur,motif de sortie prématurée LIB
1,01-001,0,NaN,22/11/2013,1,NaN,18/03/2014,MOREAU CAROLINE,Patient non opéré
2,01-002,0,NaN,20/02/2014,6,SUSPICION DE CANCER PULMONAIRE,24/03/2014,DR MOREAU,Autre
3,01-003,0,NaN,07/03/2014,1,NaN,21/03/2014,DR HOPES LUCIE,Patient non opéré
4,01-004,0,NaN,09/01/2015,6,RETRAIT DU MATERIEL ET RETRAIT DE CONSENTEMENT,09/01/2015,DEVOS,Autre


In [12]:
df_Statiques = pd.merge(df_feuille1,df_feuille2,on="SUBJID",how="left")
df_Statiques.head() 

,SUBJID,INIT_PAT,D_SCREEN,D_1ER_SYMPT,D_DIAG,D_LDOPA,D_TTT_DOPAM,D_FLUCTU_MOTR,D_FLUCTU_NONMOTR,D_DYSKINESIE,...,AGE,SEXE,SUIVI_ETUDE,D_FIN_ETUDE,D_SORTIE_PREMA,MOTIF_SORTIE_PREMA,AUTRE_PRECIS,D_INVESTIGATEUR,NOM_INVESTIGATEUR,MOTIF_SORTIE_PREMA1
0,Subject Identifier for the Study,initiales patient,date de la visite de screening,année premiers symptomes,année du diagnostic de la maladie,année d'introduction de la L-DOPA,année d'introduction du traitement dopaminergique,année d'apparition des fluctuations motrices,année d'apparition des fluctuations non motrices,année d'apparition des dyskinésies,...,âge,sexe,suivi étude,date de fin de l'étude,date de sortie prématurée,motif de sortie prématurée,autre précision,date de signature de l'investigateur,nom de l'investigateur,motif de sortie prématurée LIB
1,01-001,SR,18/11/2013,1998,1999,2000,2000,2002,2000,.D,...,67,1,0,NaN,22/11/2013,1,NaN,18/03/2014,MOREAU CAROLINE,Patient non opéré
2,01-002,TM,13/01/2014,2006,2006,2006,2006,2011,.K,.K,...,59,1,0,NaN,20/02/2014,6,SUSPICION DE CANCER PULMONAIRE,24/03/2014,DR MOREAU,Autre
3,01-003,SJ,04/03/2014,2001,2001,2003,2001,2004,.K,2004,...,61,1,0,NaN,07/03/2014,1,NaN,21/03/2014,DR HOPES LUCIE,Patient non opéré
4,01-004,DJ,12/05/2014,1998,2000,2003,2000,2003,2003,2003,...,65,2,0,NaN,09/01/2015,6,RETRAIT DU MATERIEL ET RETRAIT DE CONSENTEMENT,09/01/2015,DEVOS,Autre


In [14]:
df_Statiques.to_excel("Output/version_2/info.xlsx",index=False)